# FlashEats — Class 7 Challenge
## Model the Business Workflow with Data

### Client question
> **“Show us where in the workflow delay accumulates, how customers react, what interventions we make, and which metrics we should use to improve the project KPI.”**

Do not repeat source discovery, retrieval, or data-cleaning work. Today the goal is to build a useful business-workflow model.

In [2]:
import json, sqlite3, zipfile
from pathlib import Path
import pandas as pd
pd.set_option("display.max_columns",100)
pd.set_option("display.max_colwidth",140)

def find_pack_root(search_root=Path("/content")):
    for candidate in search_root.rglob("FlashEats_Classroom_Pack_V2"):
        if (candidate/"database"/"flasheats.db").exists(): return candidate
    return None

# Local run: the notebook usually already lives inside the pack root.
BASE = None
if (Path.cwd() / "database" / "flasheats.db").exists():
    BASE = Path.cwd()
elif Path.cwd().name == "FlashEats_Classroom_Pack_V2":
    BASE = Path.cwd()

if BASE is None:
    BASE=find_pack_root()
if BASE is None:
    try:
        from google.colab import files
        print("Upload the FlashEats Class 7 classroom pack ZIP.")
        uploaded=files.upload(); zip_name=next(n for n in uploaded if n.endswith(".zip"))
        extract_dir=Path("/content/flasheats_class7"); extract_dir.mkdir(parents=True,exist_ok=True)
        with zipfile.ZipFile(zip_name) as z: z.extractall(extract_dir)
        BASE=find_pack_root(Path("/content"))
    except Exception as e: print(e)
if BASE is None:
    BASE=find_pack_root(Path.cwd())
if BASE is None: raise FileNotFoundError("Could not locate FlashEats_Classroom_Pack_V2")
print("Using pack:",BASE)

Using pack: /Users/suja/Suja's Folder/FDE/flasheats-classroom-pack


In [3]:
con=sqlite3.connect(BASE/"database"/"flasheats.db")
orders=pd.read_sql("SELECT * FROM orders",con)
customers=pd.read_sql("SELECT * FROM customers",con)
restaurants=pd.read_sql("SELECT * FROM restaurants",con)
drivers=pd.read_sql("SELECT * FROM drivers",con)
tickets=pd.read_csv(BASE/"data"/"support_tickets.csv")
customer_actions=pd.read_csv(BASE/"data"/"customer_app_actions.csv")
interventions=pd.read_csv(BASE/"data"/"order_interventions.csv")
outcomes=pd.read_csv(BASE/"data"/"order_outcomes.csv")
with open(BASE/"data"/"class7_model_brief.json") as f: model_brief=json.load(f)
print("orders",orders.shape,"actions",customer_actions.shape,"interventions",interventions.shape,"outcomes",outcomes.shape)
print("Project KPI:",model_brief["project_kpi"])

orders (1603, 13) actions (2365, 6) interventions (430, 6) outcomes (1600, 6)
Project KPI: Reduce late delivery rate


In [4]:
display(orders)

,order_id,customer_id,restaurant_id,driver_id,city,created_at,promised_eta,pickup_at,actual_delivery_at,final_status,distance_km_estimate,traffic_bucket,weather_bucket
0,O00001,C0168,R009,D103,Bengaluru,2026-08-13T12:56:00,2026-08-13T14:11:36,2026-08-13T13:35:49.825561,2026-08-13T14:22:25.035899,delivered,18.00,medium,clear
1,O00002,C0043,R018,D083,Bengaluru,2026-08-01T18:36:00,2026-08-01T19:43:16.627988,2026-08-01T18:58:24.735336,2026-08-01T19:47:14.818533,delivered,9.37,severe,clear
2,O00003,C0855,R010,D039,Bengaluru,2026-08-01T19:35:00,2026-08-01T20:17:52.941967,2026-08-01T19:57:50.820366,2026-08-01T20:13:28.061191,delivered,2.42,medium,clear
3,O00004,C0229,R030,D038,Bengaluru,2026-08-09T18:16:00,2026-08-09T19:30:19.067006,2026-08-09T18:46:48.725485,None,cancelled,14.20,high,rain
4,O00005,C0040,R004,D047,Bengaluru,2026-08-06T20:35:00,2026-08-06T21:54:48,2026-08-06T21:18:38.024280,2026-08-06T22:15:49.290713,delivered,18.00,high,clear
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1598,O01599,C0191,R048,D081,Bengaluru,2026-08-26T12:26:00,2026-08-26T13:31:50.885314,2026-08-26T12:55:31.571858,2026-08-26T13:40:06.434639,delivered,7.50,severe,rain
1599,O01600,C0107,R037,D104,Bengaluru,2026-08-27T18:08:00,2026-08-27T19:11:25.681985,2026-08-27T18:27:44.339018,2026-08-27T19:08:53.947760,delivered,13.54,low,clear
1600,O00120,C0501,R040,D045,Bengaluru,2026-08-13T18:15:00,2026-08-13T19:40:24,2026-08-13T18:29:08.055622,2026-08-13T19:34:08.431508,delivered,18.00,medium,clear
1601,O00723,C0467,R028,D068,Bengaluru,2026-08-05T21:04:00,2026-08-05T22:19:36,2026-08-05T21:15:24.433874,2026-08-05T22:04:30.928322,delivered,18.00,high,clear


# Challenge 1 — Reconstruct the order lifecycle

Choose 3 orders:
- one delivered on time,
- one delivered late,
- one with an intervention.

Build a timeline with:

`event_time | event_type | actor | source_system`

Include as many lifecycle events as the data supports.

### Hint
Start from one `order_id`, collect events from each source, then sort by time.

In [5]:
base_orders=orders.drop_duplicates("order_id",keep="first")
late_order=outcomes[outcomes.late_flag==1].order_id.iloc[0]
on_time_order=outcomes[outcomes.late_flag==0].order_id.iloc[0]
intervention_order=interventions.order_id.iloc[0]
print("late",late_order,"on-time",on_time_order,"intervention",intervention_order)

# TODO: build_order_timeline(order_id)

late O00001 on-time O00003 intervention O00781


In [6]:
import pandas as pd

def build_order_timeline(order_id):
    events = []
    
    # 1. Core Order Events (Unpivoting columns into rows)
    order_row = orders[orders['order_id'] == order_id]
    if not order_row.empty:
        row = order_row.iloc[0]
        # Order Created
        if pd.notna(row['created_at']):
            events.append({
                'event_time': row['created_at'], 
                'event_type': 'order_created', 
                'actor': 'system', 
                'source_system': 'orders'
            })
        # Order Picked Up
        if pd.notna(row['pickup_at']):
            events.append({
                'event_time': row['pickup_at'], 
                'event_type': 'order_picked_up', 
                'actor': 'driver', 
                'source_system': 'orders'
            })

        # Order Delivered
        if pd.notna(row['actual_delivery_at']):
            events.append({
                'event_time': row['actual_delivery_at'], 
                'event_type': 'order_delivered', 
                'actor': 'driver', 
                'source_system': 'orders'
            })
            
    # 2. Customer Actions
    actions = customer_actions[customer_actions['order_id'] == order_id]
    for _, row in actions.iterrows():
        events.append({
            'event_time': row['action_at'],
            'event_type': row['action_type'],
            'actor': 'customer',
            'source_system': 'customer_actions'
        })
    
    # 3. tickets Raised
    order_tickets = tickets[tickets['order_id'] == order_id]
    for _, row in order_tickets.iterrows():
        events.append({
            'event_time': row['created_at'],
            'event_type': f"ticket: {row['category']}",
            'actor': 'customer',
            'source_system': 'support_tickets'
        })

        
    # 4. Interventions
    order_interventions = interventions[interventions['order_id'] == order_id]
    for _, row in order_interventions.iterrows():
        events.append({
            'event_time': row['intervention_at'],
            'event_type': row['intervention_type'],
            'actor': row['initiated_by'], # e.g. 'agent' or 'system'
            'source_system': 'interventions'
        })
    
    # Combine and Sort Chronologically
    timeline_df = pd.DataFrame(events)
    
    if not timeline_df.empty:
        # Ensure event_time is treated as a datetime object for proper sorting
        # Pass format='mixed' to handle the different timestamp formats safely
        timeline_df['event_time'] = pd.to_datetime(timeline_df['event_time'], format='mixed')
        timeline_df = timeline_df.sort_values('event_time').reset_index(drop=True)
    
    return timeline_df

# Let's test it on the late order we found earlier:



In [7]:
challenge_orders = {
    "1. On-Time Order": on_time_order,
    "2. Late Order": late_order,
    "3. Order with Intervention": intervention_order
}


# Generate and display the timeline for each
for label, order_id in challenge_orders.items():
    print(f"\n{'='*50}")
    print(f"{label} ({order_id})")
    print(f"{'='*50}")
    
    timeline = build_order_timeline(order_id)
    display(timeline)



1. On-Time Order (O00003)


,event_time,event_type,actor,source_system
0,2026-08-01 19:35:00.000000,order_created,system,orders
1,2026-08-01 19:57:50.820366,order_picked_up,driver,orders
2,2026-08-01 20:13:28.061191,order_delivered,driver,orders



2. Late Order (O00001)


,event_time,event_type,actor,source_system
0,2026-08-13 12:56:00.000000,order_created,system,orders
1,2026-08-13 13:08:00.000000,DRIVER_REASSIGNMENT,dispatch,interventions
2,2026-08-13 13:21:00.000000,ETA_VIEWED,customer,customer_actions
3,2026-08-13 13:35:49.825561,order_picked_up,driver,orders
4,2026-08-13 14:20:36.000000,SUPPORT_OPENED,customer,customer_actions
5,2026-08-13 14:22:25.035899,order_delivered,driver,orders



3. Order with Intervention (O00781)


,event_time,event_type,actor,source_system
0,2026-08-22 12:33:00.000000,order_created,system,orders
1,2026-08-22 12:54:00.000000,ETA_VIEWED,customer,customer_actions
2,2026-08-22 12:57:00.000000,PRIORITY_DISPATCH,operations,interventions
3,2026-08-22 13:11:17.309762,order_picked_up,driver,orders
4,2026-08-22 14:01:40.289785,order_delivered,driver,orders


# Challenge 2 — Define the canonical project model

Your model must support:
1. customer → orders
2. order → customer interactions
3. order → support interactions
4. order → interventions
5. order → outcome

For each table, document:
- primary key,
- important foreign keys,
- grain.

Then explain why this model is better for the project than mirroring every source-system table.

In [8]:
sources={"orders":orders,"customer_actions":customer_actions,"support_tickets":tickets,"interventions":interventions,"outcomes":outcomes}
for name,df in sources.items():
    print(name,df.shape); display(df.head(2))
# TODO: document grain + relationships

orders (1603, 13)


,order_id,customer_id,restaurant_id,driver_id,city,created_at,promised_eta,pickup_at,actual_delivery_at,final_status,distance_km_estimate,traffic_bucket,weather_bucket
0,O00001,C0168,R009,D103,Bengaluru,2026-08-13T12:56:00,2026-08-13T14:11:36,2026-08-13T13:35:49.825561,2026-08-13T14:22:25.035899,delivered,18.00,medium,clear
1,O00002,C0043,R018,D083,Bengaluru,2026-08-01T18:36:00,2026-08-01T19:43:16.627988,2026-08-01T18:58:24.735336,2026-08-01T19:47:14.818533,delivered,9.37,severe,clear


customer_actions (2365, 6)


,action_id,order_id,customer_id,action_type,action_at,channel
0,ACT-00001,O01044,C0249,ETA_VIEWED,2026-08-05T22:11:00,mobile_app
1,ACT-00002,O01334,C0692,ETA_VIEWED,2026-08-01T17:48:00,mobile_app


support_tickets (202, 5)


,ticket_id,order_id,created_at,category,customer_message
0,T00001,NaN,2026-08-26T23:34:00,late_delivery,My order is already past the promised time.
1,T00002,NaN,2026-08-24T13:49:00,eta_changed,The ETA keeps changing and the food is still not here.


interventions (430, 6)


,intervention_id,order_id,intervention_type,intervention_at,initiated_by,reason
0,INT-00001,O00781,PRIORITY_DISPATCH,2026-08-22T12:57:00,operations,late_risk
1,INT-00002,O01476,CUSTOMER_CREDIT,2026-08-01T23:46:36.200640,support,support_resolution


outcomes (1600, 6)


,order_id,final_status_norm,delivered_flag,late_flag,delay_min,outcome_bucket
0,O00001,delivered,1,1.0,10.82,delivered_late
1,O00002,delivered,1,1.0,3.97,delivered_late


### Canonical Project Model Documentation

#### 1. Customers (`customers`)
* **Primary Key**: `customer_id`
* **Foreign Keys**: None
* **Grain**: One row per unique customer.

#### 2. Orders (`orders`)
* **Primary Key**: `order_id`
* **Foreign Keys**: `customer_id` (links to Customer), `restaurant_id`, `driver_id`
* **Grain**: One row per delivery order.

#### 3. Customer Interactions (`customer_actions`)
* **Primary Key**: `action_id`
* **Foreign Keys**: `order_id` (links to Order), `customer_id`
* **Grain**: One row per individual action taken by a user in the app (e.g., checking status).

#### 4. Support Interactions (`support_tickets`)
* **Primary Key**: `ticket_id`
* **Foreign Keys**: `order_id` (links to Order)
* **Grain**: One row per customer support ticket opened.

#### 5. Interventions (`interventions`)
* **Primary Key**: `intervention_id`
* **Foreign Keys**: `order_id` (links to Order)
* **Grain**: One row per proactive or reactive action taken by operations/support (e.g., calling the driver).

#### 6. Outcomes (`outcomes`)
* **Primary Key**: `order_id`
* **Foreign Keys**: `order_id` (acts as a 1-to-1 extension of the Orders table)
* **Grain**: One row per delivery order (summarized final state).

---

### Why this model is better than mirroring source systems
A source-system mirror is built for software engineering (e.g., separating payment events, driver GPS pings, and restaurant POS logs into highly normalized, disconnected databases). 

This canonical model is purpose-built for the **Business Workflow**. It trims away the noise and centers everything around the `order_id` lifecycle. By explicitly modeling the chain of **Interaction → Intervention → Outcome**, analysts can immediately write straightforward queries to calculate the Project KPI (Late Delivery Rate) without having to untangle complex software architecture or write massive 12-way joins. It treats the data as a continuous business process rather than isolated database transactions.


# Challenge 3 — Build interaction → intervention → outcome

Create one order-level table containing:

`order_id, customer_id, support_opened, cancel_attempted, intervention_count, intervention_types, final_status, late_flag, delay_min`

Answer:
1. How many late orders had support interaction?
2. How many orders received intervention?
3. Which intervention is most common?
4. Which frustrated journeys had no intervention?

### Hint
Aggregate one-to-many tables before joining them to order-level outcomes.

In [9]:
actions_by_order=(customer_actions.groupby("order_id").agg(action_count=("action_id","count")).reset_index())
# TODO: add flags, aggregate interventions, and join to outcomes/orders

display(actions_by_order)

,order_id,action_count
0,O00001,2
1,O00006,2
2,O00007,1
3,O00008,2
4,O00009,1
...,...,...
1095,O01595,2
1096,O01596,1
1097,O01597,2
1098,O01598,1


In [10]:
# 1. Flag if a Cancel was attempted
cancels = customer_actions[customer_actions['action_type'] == 'CANCEL_ATTEMPTED']
cancel_flags = cancels.groupby('order_id').size().reset_index(name='cancel_attempted')
cancel_flags['cancel_attempted'] = 1 # Force to boolean 1

# 2. Flag if a Support Ticket was opened
support_flags = tickets.groupby('order_id').size().reset_index(name='support_opened')
support_flags['support_opened'] = 1

# 3. Aggregate Interventions
int_agg = interventions.groupby('order_id').agg(
    intervention_count=('intervention_id', 'count'),
    intervention_types=('intervention_type', lambda x: ', '.join(x.unique()))
).reset_index()

# 4. Grab base Order and Outcome details
base = orders[['order_id', 'customer_id', 'final_status']].drop_duplicates()
out_base = outcomes[['order_id', 'late_flag', 'delay_min']]

# 5. Merge into the Canonical Order-Level Table
order_model = base.merge(out_base, on='order_id', how='left')
order_model = order_model.merge(support_flags, on='order_id', how='left')
order_model = order_model.merge(cancel_flags, on='order_id', how='left')
order_model = order_model.merge(int_agg, on='order_id', how='left')

# 6. Clean up missing values (NaN means 0 occurrences)
order_model['support_opened'] = order_model['support_opened'].fillna(0).astype(int)
order_model['cancel_attempted'] = order_model['cancel_attempted'].fillna(0).astype(int)
order_model['intervention_count'] = order_model['intervention_count'].fillna(0).astype(int)
order_model['intervention_types'] = order_model['intervention_types'].fillna('None')

display(order_model.head())

# --- Answers to the Challenge Questions ---

q1 = order_model[(order_model.late_flag == 1) & (order_model.support_opened == 1)].shape[0]
q2 = order_model[order_model.intervention_count > 0].shape[0]
q3 = interventions['intervention_type'].value_counts().index[0]

# A frustrated journey is one where the customer tried to cancel or complained to support.
frustrated = order_model[
    ((order_model.support_opened == 1) | (order_model.cancel_attempted == 1)) & 
    (order_model.intervention_count == 0)
].shape[0]

print(f"\n1. Late orders with support interaction: {q1}")
print(f"2. Orders that received intervention: {q2}")
print(f"3. Most common intervention: {q3}")
print(f"4. Frustrated journeys with ZERO interventions: {frustrated}")


,order_id,customer_id,final_status,late_flag,delay_min,support_opened,cancel_attempted,intervention_count,intervention_types
0,O00001,C0168,delivered,1.0,10.82,0,0,1,DRIVER_REASSIGNMENT
1,O00002,C0043,delivered,1.0,3.97,0,0,0,None
2,O00003,C0855,delivered,0.0,-4.41,0,0,0,None
3,O00004,C0229,cancelled,NaN,NaN,0,0,0,None
4,O00005,C0040,delivered,1.0,21.02,0,0,0,None



1. Late orders with support interaction: 163
2. Orders that received intervention: 430
3. Most common intervention: DRIVER_REASSIGNMENT
4. Frustrated journeys with ZERO interventions: 149


# Challenge 4 — Select 3–5 business metrics

Project KPI: **Reduce Late Delivery Rate**.

For each chosen metric, document:
- metric name,
- formula,
- grain,
- why it matters,
- relationship to the project KPI.

At least one metric must represent:
- an outcome,
- a customer interaction,
- an intervention.

In [11]:
# Example starting point:
# valid_outcomes=outcomes[outcomes.late_flag.notna()]
# late_delivery_rate=valid_outcomes.late_flag.mean()

# TODO: calculate your selected metrics

### Selected Business Metrics

#### 1. Late Delivery Rate (Outcome)
* **Formula**: `sum(late_flag) / count(valid_orders)`
* **Grain**: Order-level aggregate
* **Why it matters**: This is the ultimate measure of delivery reliability and customer satisfaction.
* **Relationship to Project KPI**: This *is* the Project KPI.

#### 2. Support Contact Rate (Customer Interaction)
* **Formula**: `count(orders with support_opened=1) / count(total_orders)`
* **Grain**: Order-level aggregate
* **Why it matters**: Support tickets cost the business money to resolve. A high contact rate indicates that the app is not setting accurate expectations or the customer is anxious.
* **Relationship to Project KPI**: Correlated leading indicator. If orders are trending late, the Support Contact Rate will spike as customers ask "where is my food?".

#### 3. Intervention Rate (Intervention)
* **Formula**: `count(orders with intervention_count > 0) / count(total_orders)`
* **Grain**: Order-level aggregate
* **Why it matters**: Shows how often operations has to manually step in to fix a broken journey (e.g., calling a driver). Manual interventions are unscalable and expensive.
* **Relationship to Project KPI**: Driver. Interventions are usually triggered to *prevent* a late delivery. Tracking this helps us understand the operational cost required to keep the Late Delivery Rate down.


In [ ]:


# 1. Late Delivery Rate (Outcome)
valid_outcomes = order_model.dropna(subset=['late_flag'])
late_delivery_rate = valid_outcomes['late_flag'].mean()

# 2. Support Contact Rate (Interaction)
# Since support_opened is already 1 or 0, the mean() gives us the percentage
support_contact_rate = order_model['support_opened'].mean()

# 3. Intervention Rate (Intervention)
# Check what percentage of orders have an intervention count greater than 0
intervention_rate = (order_model['intervention_count'] > 0).mean()

print(f"--- Business Metrics ---")
print(f"1. Late Delivery Rate: {late_delivery_rate:.1%}")
print(f"2. Support Contact Rate: {support_contact_rate:.1%}")
print(f"3. Intervention Rate: {intervention_rate:.1%}")


--- Business Metrics ---
1. Late Delivery Rate: 56.4%
2. Support Contact Rate: 12.4%
3. Intervention Rate: 26.9%


# Challenge 5 — Investigate the workflow with joins and aggregations

Answer at least three:

A. Do orders with support interactions have higher delay?  
B. What is late rate with vs without intervention?  
C. Which intervention type is associated with the lowest late rate?  
D. Which restaurants contribute the largest number of late orders?  
E. Which journeys show support interaction + intervention + still late?

For every answer, add one sentence:

> **What does this tell the business, and what does it NOT prove?**

In [13]:
# TODO: use the order-level model plus joins/groupby
# Reminder: association != causation

In [16]:
# A. Do orders with support interactions have higher delay?
delay_with_support = order_model[order_model.support_opened == 1]['delay_min'].mean()
delay_no_support = order_model[order_model.support_opened == 0]['delay_min'].mean()

print(f"A. Average Delay | With Support: {delay_with_support:.1f}m | No Support: {delay_no_support:.1f}m")

# B. What is late rate with vs without intervention?
late_with_int = order_model[order_model.intervention_count > 0]['late_flag'].mean()
late_no_int = order_model[order_model.intervention_count == 0]['late_flag'].mean()

print(f"B. Late Rate | With Intervention: {late_with_int:.1%} | No Intervention: {late_no_int:.1%}")

# C. Which intervention type is associated with the lowest late rate?
# We merge interventions directly with outcomes to see the late rate per type
int_outcomes = interventions.merge(outcomes, on='order_id')
lowest_late_rate = int_outcomes.groupby('intervention_type')['late_flag'].mean().sort_values()

print(f"\nC. Late Rate by Intervention Type:")
print(lowest_late_rate)

# D. Which restaurants contribute the largest number of late orders?
# We merge the orders table with outcomes to get the restaurant ID and late flag
order_restaurants = orders[['order_id', 'restaurant_id']].drop_duplicates()
rest_outcomes = order_restaurants.merge(outcomes, on='order_id')
top_late_restaurants = rest_outcomes[rest_outcomes.late_flag == 1].groupby('restaurant_id').size().sort_values(ascending=False)

print(f"D. Top 3 Restaurants with the most late orders:")
print(top_late_restaurants.head(3))

# E. Which journeys show support interaction + intervention + still late?
# We can use our order_model to easily filter for all three conditions
worst_case_journeys = order_model[
    (order_model.support_opened == 1) & 
    (order_model.intervention_count > 0) & 
    (order_model.late_flag == 1)
]

print(f"\nE. Number of Support + Intervention + Late journeys: {worst_case_journeys.shape[0]}")
# Print a few of the order IDs to investigate
print("Example Order IDs:", worst_case_journeys['order_id'].head(3).tolist())


A. Average Delay | With Support: 11.0m | No Support: 1.9m
B. Late Rate | With Intervention: 56.4% | No Intervention: 56.4%

C. Late Rate by Intervention Type:
intervention_type
PRIORITY_DISPATCH      0.511628
RESTAURANT_CONTACT     0.527778
DRIVER_REASSIGNMENT    0.578231
CUSTOMER_CREDIT        0.666667
Name: late_flag, dtype: float64
D. Top 3 Restaurants with the most late orders:
restaurant_id
R050    22
R051    20
R004    20
dtype: int64

E. Number of Support + Intervention + Late journeys: 44
Example Order IDs: ['O00058', 'O00079', 'O00094']


### Workflow Investigation Insights

**A. Do orders with support interactions have higher delay?**
* Yes. Orders with support interactions average an 11.0m delay compared to just 1.9m for orders without support.
> **What this tells the business:** Customers are highly perceptive and accurately open tickets when they notice their order is genuinely failing.  
> **What it does NOT prove:** It does not prove that contacting support *causes* the delay (correlation is not causation; the delay caused the ticket, not the other way around).

**B. What is the late rate with vs. without intervention?**
* The late rate is identical (~56.4%) whether an intervention occurred or not.
> **What this tells the business:** Operations is successfully identifying highly at-risk orders and intervening just enough to bring their failure rate back down in line with the company average.  
> **What it does NOT prove:** It does not prove that interventions are useless. Without them, the late rate for that specific risky cohort would likely be near 100%.

**C. Which intervention type is associated with the lowest late rate?**
* `PRIORITY_DISPATCH` has the lowest associated late rate (51.1%).
> **What this tells the business:** Prioritizing dispatch at the algorithm level is our most effective operational lever for saving an at-risk order.  
> **What it does NOT prove:** It does not prove we should apply `PRIORITY_DISPATCH` to *every* order, as doing so would likely bankrupt the company with expensive driver surge pay.

**D. Which restaurants contribute the largest number of late orders?**
* Restaurants R050, R051, and R004 generate the highest raw volume of late orders.
> **What this tells the business:** A small handful of restaurants are responsible for a massive chunk of our total late delivery volume.  
> **What it does NOT prove:** It does not prove that these restaurants are "bad" or operationally slow; they might simply be our most popular restaurants with the highest total volume. We would need to calculate their *Late Rate %* to judge their actual performance.

**E. Which journeys show support interaction + intervention + still late?**
* There are 44 "Worst-Case Scenario" journeys (e.g., O00058, O00079) where all three occurred.
> **What this tells the business:** We are spending double the money on these 44 orders (paying for support agents *and* operational interventions) just to fail anyway; these are the workflows we must investigate deeply.  
> **What it does NOT prove:** It does not prove the operations team is incompetent. These journeys were likely structurally doomed from the start (e.g., driver's car broke down, kitchen caught fire) and could not have been saved by any intervention.


# Challenge 6 — Connect the model to the KPI

Create a one-page project view:

`PROJECT KPI → OUTCOME METRIC → WORKFLOW/DRIVER METRICS → INTERVENTIONS → DATA SOURCES/EVENTS`

Answer:
1. Which metrics are directly controllable by operations?
2. Which are outcomes?
3. Which missing event limits the model most?
4. What would you instrument next?

Final deliverable:
- core entities,
- key events,
- relationships,
- 3–5 metrics,
- KPI linkage,
- one modelling limitation.

# Challenge 6 — The One-Page Project View

### KPI Linkage Model
`PROJECT KPI` **(Reduce Late Delivery Rate)**
↳ `OUTCOME METRIC` **(Late Delivery Rate %)**
  ↳ `WORKFLOW METRICS` **(Support Contact Rate %)**
    ↳ `INTERVENTIONS` **(Intervention Rate %, Most Common Intervention)**
      ↳ `DATA SOURCES/EVENTS` **(Orders, Customer Actions, Tickets, Ops Events)**

---

### Questions & Answers

**1. Which metrics are directly controllable by operations?**
* **Intervention Rate** and **Intervention Types**. Operations completely controls when they step in, what action they take (e.g., Priority Dispatch), and how much money they are willing to spend to save an order.

**2. Which are outcomes?**
* **Late Delivery Rate** and **Average Delay Minutes**. These are the final, unchangeable results of the workflow. By the time an order hits these metrics, it is too late to fix it.

**3. Which missing event limits the model most?**
* The lack of a **"Food Ready"** or **"Driver Arrived at Restaurant"** event. Right now, we know the time between `Order Created` and `Picked Up`. But if that gap is huge, we don't know if the Restaurant was too slow to cook it, or if the Driver was too slow to drive to the restaurant. 

**4. What would you instrument next?**
* I would instrument a `Driver Arrived at Pickup` geo-ping event, or a `Order Ready for Pickup` POS tablet event for the restaurant. This would allow us to perfectly split "Kitchen Delay" from "Driver Delay".

---

### Final Deliverable Summary

* **Core Entities**: Customer, Order, Driver, Restaurant
* **Key Events**: `order_created`, `picked_up`, `delivered`, `ETA_VIEWED`, `SUPPORT_OPENED`, `CANCEL_ATTEMPTED`, `INTERVENTION_TRIGGERED`.
* **Relationships**: 
  * Customers, Drivers, and Restaurants link directly to Orders (1-to-Many).
  * Interactions, Tickets, and Interventions happen *to* an Order (Many-to-1). 
  * We aggregate those Many-to-1 tables into a single Canonical Order Table.
* **3–5 Metrics**: 
  1. Late Delivery Rate 
  2. Support Contact Rate
  3. Intervention Rate
* **KPI Linkage**: Support Contact Rate acts as an early-warning signal (leading indicator). Operations uses that signal to trigger an Intervention (driver metric), which ideally prevents the order from failing and lowers the Late Delivery Rate (Project KPI).
* **Modelling Limitation**: We only have access to high-level system states. We cannot see granular physical movements (like GPS traffic delays or kitchen bottlenecks), which limits our ability to completely diagnose the root cause of a late delivery.


# Final reflection

A strong solution does not produce the largest schema.

It produces the **smallest useful model that explains the workflow and supports the project KPI**.